# 🏦 EXP-09 v3 — Domain-Specific Embeddings Benchmark
### Banking & Finance RAG Retrieval Evaluation with Advanced Chunking

---

## What's new in v3 (vs v2)

| Change | Details |
|--------|---------|
| **Professional Dataset** | Full-text research papers from [arXiv Quantitative Finance](https://arxiv.org/list/q-fin/recent) — Real academic papers on banking, portfolio management, risk management, derivatives, and computational finance |
| **Advanced Chunking** | Two strategies implemented: **RecursiveCharacterTextSplitter** (paragraph→sentence→word cascade) and **SemanticChunker** (similarity-driven sentence merging). Chunks (not full docs) are indexed. |
| **FinBERT Fallback Chain** | `yiyanghkust/finbert-tone` → `ProsusAI/finbert` → `nickmuchi/financial-news-distilroberta-base`. Each fallback is labelled in results. |
| **Model D fixed** | Confirmed working: `nickmuchi/financial-news-distilroberta-base` replaces broken `nickmuchi/finance-embeddings-investopedia` |
| **Richer Corpus** | 25 topic documents → chunked to ~80–120 chunks; more realistic RAG retrieval scenario |
| **Metrics** | Recall@1, Recall@3, Recall@5, MRR — all reported at chunk level mapped back to source document |
| **Summary Report** | Structured `.md` summary report with interpretation, model ranking table, and chunking analysis |

---
> **Runtime:** Go to **Runtime → Change runtime type → T4 GPU** before running.


In [ ]:
# CELL 1 — Install Dependencies
!pip install -q chromadb sentence-transformers datasets nltk kaggle
!pip install -q numpy matplotlib

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print('\n✅ All dependencies installed.')


In [ ]:
# CELL 2 — GPU Check
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  CPU only — Runtime → Change runtime type → T4 GPU recommended.')
print(f'   PyTorch: {torch.__version__}')


In [ ]:
# CELL 3 — Configuration & Model Registry
import os, json, time, textwrap, re
from pathlib import Path
from datetime import datetime
import numpy as np

# ── Chunking Config ──────────────────────────────────────────────────────────
CHUNK_STRATEGY = 'semantic'    # 'recursive' | 'semantic'
CHUNK_SIZE     = 300           # target chars per chunk (recursive)
CHUNK_OVERLAP  = 60            # overlap chars (recursive)
SEM_THRESHOLD  = 0.75          # cosine sim below this → break chunk (semantic)
MAX_CHUNK_CHARS = 500          # hard ceiling per chunk

# ── Model Registry ───────────────────────────────────────────────────────────
# Each model lists a primary HF ID and an ordered fallback chain.
# SentenceTransformer loads the first that succeeds.
MODELS = {
    'A_MiniLM': {
        'hf_chain' : ['sentence-transformers/all-MiniLM-L6-v2'],
        'label'    : 'all-MiniLM-L6-v2 (General baseline)',
        'collection': 'exp09v3_minilm',
        'type'     : 'general',
    },
    'B_BGE': {
        'hf_chain' : ['BAAI/bge-small-en-v1.5', 'BAAI/bge-m3'],
        'label'    : 'BGE-small-en-v1.5 (General compact)',
        'collection': 'exp09v3_bge',
        'type'     : 'general',
    },
    'C_FinBERT': {
        # Fallback chain: finbert-tone → ProsusAI/finbert → financial-news-distilroberta
        'hf_chain' : [
            'yiyanghkust/finbert-tone',
            'ProsusAI/finbert',
            'nickmuchi/financial-news-distilroberta-base',
        ],
        'label'    : 'FinBERT-tone (Finance sentiment)',
        'collection': 'exp09v3_finbert',
        'type'     : 'domain',
    },
    'D_FinNews': {
        'hf_chain' : [
            'nickmuchi/financial-news-distilroberta-base',
            'nickmuchi/finbert-tone-finetuned-finance-topic-classification',
            'sentence-transformers/all-mpnet-base-v2',
        ],
        'label'    : 'financial-news-distilroberta (Finance domain)',
        'collection': 'exp09v3_finnews',
        'type'     : 'domain',
    },
}

TOP_K        = 5
CHROMA_PATH  = '/content/chroma_exp09v3'
OUTPUT_DIR   = Path('/content/exp09_outputs_v3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Configuration loaded.')
print(f'   Chunking : {CHUNK_STRATEGY.upper()} | size={CHUNK_SIZE} | overlap={CHUNK_OVERLAP}')
print(f'   Models   : {list(MODELS.keys())}')
print(f'   Outputs  : {OUTPUT_DIR}')


In [ ]:
# CELL 4 — Download Full-Text Finance Research Papers from arXiv (PDFs)
#
# DATASET: arXiv Quantitative Finance (q-fin) Full Research Papers
# ─────────────────────────────────────────────────────────────────────────────
# We download FULL PDF research papers from arXiv's q-fin category and extract
# the complete text content (not just abstracts).
#
# Topics covered:
# - Banking & Financial Institutions
# - Risk Management  
# - Portfolio Management
# - Computational Finance
# - Trading & Market Microstructure
# - Mathematical Finance
# - Economics & Econometrics
#
# Source: https://arxiv.org/list/q-fin/recent
# ─────────────────────────────────────────────────────────────────────────────

import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
import re
from time import sleep
import os

print('⬇️  Downloading full-text finance research papers from arXiv...')
print('   This will download PDFs and extract full text content.')
print('   This may take 3-5 minutes...')
print()

# Install PDF extraction library
print('📦 Installing PDF text extraction library...')
import subprocess
subprocess.run(['pip', 'install', '-q', 'PyPDF2'], check=True)
import PyPDF2
print('   ✓ PDF library installed')
print()

# arXiv API endpoint
ARXIV_API = 'http://export.arxiv.org/api/query?'

# Define finance topics - using more specific queries for better papers
FINANCE_TOPICS = {
    'Banking & Credit Risk': 'cat:q-fin.RM AND (bank OR credit OR lending)',
    'Portfolio Management': 'cat:q-fin.PM AND (portfolio OR asset)',
    'Risk Management': 'cat:q-fin.RM AND (risk OR VaR)',
    'Derivatives Pricing': 'cat:q-fin.PR AND (option OR derivative)',
    'Market Microstructure': 'cat:q-fin.TR AND (trading OR market)',
    'Corporate Finance': 'cat:q-fin.GN AND (corporate OR valuation)',
    'Econometrics': 'cat:q-fin.ST AND (econometric OR forecast)',
    'Computational Finance': 'cat:q-fin.CP AND (computational OR algorithm)',
}

def fetch_arxiv_papers(query, max_results=10):
    """Fetch papers from arXiv API"""
    params = {
        'search_query': query,
        'start': 0,
        'max_results': max_results,
        'sortBy': 'relevance',
        'sortOrder': 'descending'
    }
    
    url = ARXIV_API + urllib.parse.urlencode(params)
    
    try:
        with urllib.request.urlopen(url) as response:
            return response.read()
    except Exception as e:
        print(f'  ⚠️  Error fetching papers: {e}')
        return None

def parse_arxiv_response(xml_data):
    """Parse arXiv API XML response"""
    papers = []
    
    try:
        root = ET.fromstring(xml_data)
        ns = {'atom': 'http://www.w3.org/2005/Atom'}
        
        for entry in root.findall('atom:entry', ns):
            title = entry.find('atom:title', ns).text.strip().replace('\\n', ' ')
            summary = entry.find('atom:summary', ns).text.strip().replace('\\n', ' ')
            paper_id = entry.find('atom:id', ns).text.strip()
            
            # Get PDF link
            pdf_link = None
            for link in entry.findall('atom:link', ns):
                if link.get('title') == 'pdf':
                    pdf_link = link.get('href')
                    break
            
            # Clean up text
            title = re.sub(r'\\s+', ' ', title)
            summary = re.sub(r'\\s+', ' ', summary)
            
            papers.append({
                'title': title,
                'abstract': summary,
                'id': paper_id,
                'pdf_url': pdf_link
            })
    except Exception as e:
        print(f'  ⚠️  Error parsing XML: {e}')
    
    return papers

def download_pdf(url, filepath):
    """Download PDF from URL"""
    try:
        urllib.request.urlretrieve(url, filepath)
        return True
    except Exception as e:
        print(f'    ⚠️  PDF download failed: {e}')
        return False

def extract_text_from_pdf(pdf_path):
    """Extract text from PDF file"""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            text = ''
            for page in pdf_reader.pages:
                text += page.extract_text() + '\\n\\n'
            return text.strip()
    except Exception as e:
        print(f'    ⚠️  PDF text extraction failed: {e}')
        return None

# Create directories
papers_dir = '/content/arxiv_papers'
pdfs_dir = '/content/arxiv_pdfs'
os.makedirs(papers_dir, exist_ok=True)
os.makedirs(pdfs_dir, exist_ok=True)

# Download papers for each topic
all_papers = []
topic_counts = {}
successful_downloads = 0

for topic, query in FINANCE_TOPICS.items():
    print(f'  📄 Fetching: {topic}...')
    xml_data = fetch_arxiv_papers(query, max_results=10)
    
    if xml_data:
        papers = parse_arxiv_response(xml_data)
        print(f'     Found {len(papers)} papers')
        
        # Download PDFs and extract text
        topic_success = 0
        for i, paper in enumerate(papers[:5], 1):  # Limit to 5 per topic for speed
            if paper['pdf_url']:
                arxiv_id = paper['id'].split('/')[-1].replace(':', '_')
                pdf_path = os.path.join(pdfs_dir, f'{arxiv_id}.pdf')
                
                print(f'     [{i}/5] Downloading PDF: {arxiv_id}...', end=' ')
                
                if download_pdf(paper['pdf_url'], pdf_path):
                    print('✓ Extracting text...', end=' ')
                    full_text = extract_text_from_pdf(pdf_path)
                    
                    if full_text and len(full_text) > 1000:  # Ensure substantial content
                        paper['text'] = full_text
                        paper['full_text'] = full_text
                        all_papers.append(paper)
                        topic_success += 1
                        successful_downloads += 1
                        print('✓')
                    else:
                        print('⚠️  (insufficient text)')
                else:
                    print('✗')
            
            sleep(0.5)  # Be nice to arXiv
        
        topic_counts[topic] = topic_success
    
    sleep(1)

print()
print(f'💾 Saving {successful_downloads} full-text papers to {papers_dir}/')

for i, paper in enumerate(all_papers, 1):
    arxiv_id = paper['id'].split('/')[-1].replace(':', '_')
    filename = f'paper_{i:03d}_{arxiv_id}.txt'
    filepath = os.path.join(papers_dir, filename)
    
    # Save full paper text
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(f"Title: {paper['title']}\\n")
        f.write(f"arXiv ID: {paper['id']}\\n")
        f.write(f"PDF URL: {paper['pdf_url']}\\n")
        f.write(f"\\n{'='*80}\\n")
        f.write(f"FULL RESEARCH PAPER TEXT\\n")
        f.write(f"{'='*80}\\n\\n")
        f.write(paper['full_text'])

print(f'   ✓ Saved {len(all_papers)} full-text research papers')
print(f'   📁 Text files: {papers_dir}/')
print(f'   📁 PDF files: {pdfs_dir}/')
print(f'   (Both visible in Colab file menu on the left)')
print()
print(f'✅ Downloaded {len(all_papers)} complete finance research papers')
print(f'   Average paper length: {int(sum(len(p["full_text"]) for p in all_papers) / len(all_papers)):,} characters')
print()
print('── Papers by topic ──')
for topic, count in topic_counts.items():
    bar = '█' * min(count, 30)
    print(f'  {bar:<30} {count:2d}  {topic}')


In [ ]:
# CELL 5 — Corpus Construction: Organize Papers into Thematic Documents
#
# Strategy:
#   1. Group papers by finance topics
#   2. Each topic becomes a document containing multiple paper abstracts
#   3. This creates a realistic corpus for RAG retrieval testing

import random
random.seed(42)

# Group papers by topic keywords
TOPIC_KEYWORDS = {
    'DOC-01': ('Banking & Credit Risk', 
               r'bank|credit risk|lending|loan|deposit|basel|capital adequacy|NPL|non.performing'),
    'DOC-02': ('Portfolio Management & Asset Allocation',
               r'portfolio|asset allocation|diversification|optimization|mean.variance|sharpe|efficient frontier'),
    'DOC-03': ('Market Microstructure & Trading',
               r'market microstructure|trading|liquidity|bid.ask|order book|high.frequency|algorithmic trading'),
    'DOC-04': ('Derivatives & Options Pricing',
               r'derivative|option|black.scholes|volatility|hedging|swap|future|forward contract'),
    'DOC-05': ('Risk Management & VaR',
               r'risk management|value at risk|VaR|CVaR|stress test|scenario analysis|risk measure'),
    'DOC-06': ('Corporate Finance & Valuation',
               r'corporate finance|valuation|DCF|WACC|capital structure|dividend|M&A|merger'),
    'DOC-07': ('Econometrics & Time Series',
               r'econometric|time series|GARCH|ARIMA|cointegration|forecasting|volatility model'),
    'DOC-08': ('Machine Learning in Finance',
               r'machine learning|neural network|deep learning|prediction|classification|regression|AI'),
    'DOC-09': ('Behavioral Finance & Market Anomalies',
               r'behavioral|anomaly|sentiment|investor psychology|bias|herding|overreaction'),
    'DOC-10': ('Cryptocurrency & Blockchain',
               r'cryptocurrency|bitcoin|blockchain|crypto|digital currency|DeFi|ethereum'),
    'DOC-11': ('ESG & Sustainable Finance',
               r'ESG|sustainable|green|climate|carbon|environmental|social|governance'),
    'DOC-12': ('Monetary Policy & Central Banking',
               r'monetary policy|central bank|interest rate|inflation|quantitative easing|federal reserve'),
    'DOC-13': ('Financial Regulation & Compliance',
               r'regulation|compliance|dodd.frank|MiFID|IFRS|accounting standard|regulatory'),
    'DOC-14': ('Systemic Risk & Financial Stability',
               r'systemic risk|financial stability|contagion|interconnectedness|too big to fail|crisis'),
    'DOC-15': ('Fixed Income & Bond Markets',
               r'bond|fixed income|yield curve|duration|credit spread|sovereign debt|treasury'),
}

def classify_paper(paper_text):
    """Classify paper into topic based on keyword matching"""
    text_lower = paper_text.lower()
    matches = []
    
    for doc_id, (topic, pattern) in TOPIC_KEYWORDS.items():
        if re.search(pattern, text_lower, re.IGNORECASE):
            # Count number of keyword matches
            match_count = len(re.findall(pattern, text_lower, re.IGNORECASE))
            matches.append((doc_id, match_count))
    
    if matches:
        # Return topic with most keyword matches
        matches.sort(key=lambda x: x[1], reverse=True)
        return matches[0][0]
    return None

# Classify papers into topics
topic_buckets = {doc_id: [] for doc_id in TOPIC_KEYWORDS}
unclassified = 0

for paper in all_papers:
    doc_id = classify_paper(paper['text'])
    if doc_id:
        topic_buckets[doc_id].append(paper)
    else:
        unclassified += 1

# Build CORPUS — only keep topics with at least 3 papers
MIN_PAPERS = 3
CORPUS = []

for doc_id, (topic, _) in TOPIC_KEYWORDS.items():
    papers = topic_buckets[doc_id]
    if len(papers) >= MIN_PAPERS:
        random.shuffle(papers)
        
        # Combine paper abstracts into a single document
        # Each paper abstract becomes a paragraph
        paragraphs = []
        for paper in papers:
            # Format: Title. Full paper text.
            para = f"Title: {paper['title']}\n\n{paper['full_text']}\n\n{'='*80}\n"
            paragraphs.append(para)
        
        text = '\n\n'.join(paragraphs)
        
        CORPUS.append({
            'id': doc_id,
            'title': topic,
            'text': text,
            'paper_count': len(papers),
            'papers': papers  # Keep reference to original papers
        })

CORPUS.sort(key=lambda x: x['id'])

print(f'✅ Corpus: {len(CORPUS)} thematic documents from {len(all_papers)} research papers')
print(f'   ({unclassified} papers unclassified or distributed across topics)')
print()
print('── Documents in corpus ──')
for d in CORPUS:
    bar = '█' * min(d['paper_count'], 30)
    print(f'  {d["id"]}  {bar:<30}  {d["paper_count"]:2d} papers  {d["title"]}')

# Save thematic documents to /content/corpus_documents/ for inspection
corpus_docs_dir = '/content/corpus_documents'
os.makedirs(corpus_docs_dir, exist_ok=True)

print()
print(f'💾 Saving thematic documents to {corpus_docs_dir}/')

for doc in CORPUS:
    filename = f'{doc["id"]}_{doc["title"].replace(" ", "_").replace("&", "and")}.txt'
    filepath = os.path.join(corpus_docs_dir, filename)
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(f"Document ID: {doc['id']}\n")
        f.write(f"Topic: {doc['title']}\n")
        f.write(f"Number of Papers: {doc['paper_count']}\n")
        f.write(f"\n{'='*80}\n")
        f.write(f"COMBINED RESEARCH PAPERS TEXT\n")
        f.write(f"{'='*80}\n\n")
        f.write(doc['text'])

print(f'   ✓ Saved {len(CORPUS)} thematic documents')
print(f'   📁 Location: {corpus_docs_dir}/')
print(f'   (These are the DOC-01, DOC-02, etc. used in evaluation)')
print()
print('── Sample document ──')
if CORPUS:
    sample = CORPUS[0]
    print(f'  [{sample["id"]}] {sample["title"]}')
    print(f'  Papers: {sample["paper_count"]}')
    print(f'  Text preview: {sample["text"][:200]}...')


In [ ]:
# CELL 6 — Advanced Chunking Implementation
#
# TWO strategies available:
#
# 1. RecursiveCharacterTextSplitter
#    — Splits on hierarchy: ["\n\n", "\n", ". ", " ", ""]
#    — Fixed window size with overlap. Fast, deterministic.
#    — Good baseline; may split mid-sentence.
#
# 2. SemanticChunker (recommended)
#    — Split text into sentences using NLTK.
#    — Compute cosine similarity between consecutive sentence embeddings.
#    — Start a new chunk wherever similarity drops below SEM_THRESHOLD.
#    — Produces semantically coherent chunks; sentence boundaries respected.

from sentence_transformers import SentenceTransformer
import nltk
from nltk.tokenize import sent_tokenize

# Load a lightweight model just for semantic chunking similarity signal
_CHUNK_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
_chunk_model = SentenceTransformer(_CHUNK_MODEL_NAME, device=device)
print(f'✅ Chunk-signal model loaded: {_CHUNK_MODEL_NAME}')


# ── 1. Recursive Character Text Splitter ─────────────────────────────────────
def recursive_split(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    separators = ['\n\n', '\n', '. ', ' ', '']
    def _split(text, seps):
        if not seps or len(text) <= chunk_size:
            return [text] if text.strip() else []
        sep, rest = seps[0], seps[1:]
        parts = text.split(sep)
        chunks, buf = [], ''
        for part in parts:
            candidate = (buf + sep + part).lstrip() if buf else part
            if len(candidate) <= chunk_size:
                buf = candidate
            else:
                if buf.strip():
                    chunks.extend(_split(buf.strip(), rest))
                buf = part
        if buf.strip():
            chunks.extend(_split(buf.strip(), rest))
        return chunks

    raw_chunks = _split(text, separators)
    # Apply overlap
    final, i = [], 0
    while i < len(raw_chunks):
        chunk = raw_chunks[i]
        if i > 0 and overlap > 0:
            prev_words = raw_chunks[i-1].split()
            overlap_text = ' '.join(prev_words[-max(1, overlap//6):])
            chunk = overlap_text + ' ' + chunk
        final.append(chunk[:MAX_CHUNK_CHARS].strip())
        i += 1
    return [c for c in final if len(c) > 30]


# ── 2. Semantic Chunker ──────────────────────────────────────────────────────
def semantic_chunk(text, threshold=SEM_THRESHOLD, max_chars=MAX_CHUNK_CHARS):
    sentences = sent_tokenize(text)
    if len(sentences) <= 2:
        return [text[:max_chars]]

    embs = _chunk_model.encode(sentences, normalize_embeddings=True,
                                show_progress_bar=False)
    # Cosine similarity between consecutive sentences
    sims = [float(np.dot(embs[i], embs[i+1])) for i in range(len(embs)-1)]

    chunks, buf = [], sentences[0]
    for i, sim in enumerate(sims):
        next_sent = sentences[i+1]
        candidate = buf + ' ' + next_sent
        if sim >= threshold and len(candidate) <= max_chars:
            buf = candidate
        else:
            if buf.strip():
                chunks.append(buf.strip())
            buf = next_sent
    if buf.strip():
        chunks.append(buf.strip())

    # Merge tiny orphan chunks (<60 chars) into previous
    merged = []
    for c in chunks:
        if merged and len(c) < 60:
            merged[-1] = merged[-1] + ' ' + c
        else:
            merged.append(c)
    return [c[:max_chars] for c in merged if len(c) > 30]


# ── Apply chosen strategy to full corpus ─────────────────────────────────────
chunker = semantic_chunk if CHUNK_STRATEGY == 'semantic' else recursive_split

CHUNKS = []  # list of dicts: id, source_doc_id, source_title, text, chunk_idx
for doc in CORPUS:
    doc_chunks = chunker(doc['text'])
    for i, chunk_text in enumerate(doc_chunks):
        CHUNKS.append({
            'chunk_id'    : f"{doc['id']}-C{i+1:02d}",
            'source_doc_id': doc['id'],
            'source_title' : doc['title'],
            'text'         : chunk_text,
            'chunk_idx'    : i,
            'total_chunks' : len(doc_chunks),
        })

# Stats
chunks_per_doc = {}
for c in CHUNKS:
    chunks_per_doc.setdefault(c['source_doc_id'], []).append(len(c['text']))

print(f'\n✅ Chunking complete  [{CHUNK_STRATEGY.upper()}]')
print(f'   Documents : {len(CORPUS)}')
print(f'   Chunks    : {len(CHUNKS)}')
print(f'   Avg chars : {np.mean([len(c["text"]) for c in CHUNKS]):.0f}')
print(f'   Max chars : {max(len(c["text"]) for c in CHUNKS)}')
print(f'   Min chars : {min(len(c["text"]) for c in CHUNKS)}')
print()
print('── Chunks per document ──')
for doc_id, char_lens in sorted(chunks_per_doc.items()):
    bar = '█' * len(char_lens)
    print(f'  {doc_id}  {bar:<15}  {len(char_lens)} chunks  '
          f'avg={int(np.mean(char_lens))} chars')
print()
print('── Sample chunk ──')
sample = CHUNKS[3]
print(f'  [{sample["chunk_id"]}] {sample["source_title"]}')
print(f'  {repr(sample["text"][:200])}...')


In [ ]:
# CELL 7 — 50 Evaluation Questions for Finance Research Papers
#
# Each question maps to 1-3 source documents (DOC-xx) based on finance topics.
# Questions are designed to test RAG retrieval on academic finance concepts.
# Multiple relevant documents test the system's ability to retrieve from multiple sources.

QUESTIONS = [
    # ── Single Document Questions ─────────────────────────────────────────
    {'id':'Q01','text':'What are the main approaches to measuring credit risk in banking?','relevant':['DOC-01']},
    {'id':'Q02','text':'How do Basel regulations impact bank capital requirements?','relevant':['DOC-01']},
    {'id':'Q03','text':'Explain the mean-variance optimization framework for portfolio construction.','relevant':['DOC-02']},
    {'id':'Q04','text':'What is the Sharpe ratio and how is it used in portfolio evaluation?','relevant':['DOC-02']},
    {'id':'Q05','text':'What role does liquidity play in market microstructure?','relevant':['DOC-03']},
    {'id':'Q06','text':'How do high-frequency trading strategies impact market quality?','relevant':['DOC-03']},
    {'id':'Q07','text':'Describe the Black-Scholes option pricing model and its assumptions.','relevant':['DOC-04']},
    {'id':'Q08','text':'What are the Greeks in options trading and why are they important?','relevant':['DOC-04']},
    {'id':'Q09','text':'What is Value at Risk (VaR) and what are its limitations?','relevant':['DOC-05']},
    {'id':'Q10','text':'Explain Conditional Value at Risk (CVaR) and how it differs from VaR.','relevant':['DOC-05']},
    
    # ── Two Document Questions ────────────────────────────────────────────
    {'id':'Q11','text':'How do credit risk models relate to portfolio optimization strategies?','relevant':['DOC-01','DOC-02']},
    {'id':'Q12','text':'What is the relationship between banking regulation and systemic risk?','relevant':['DOC-01','DOC-14']},
    {'id':'Q13','text':'How does portfolio diversification help manage market risk?','relevant':['DOC-02','DOC-05']},
    {'id':'Q14','text':'What role do derivatives play in portfolio hedging strategies?','relevant':['DOC-02','DOC-04']},
    {'id':'Q15','text':'How does market liquidity affect derivative pricing?','relevant':['DOC-03','DOC-04']},
    {'id':'Q16','text':'What is the connection between trading strategies and risk management?','relevant':['DOC-03','DOC-05']},
    {'id':'Q17','text':'How do option pricing models incorporate volatility forecasting?','relevant':['DOC-04','DOC-07']},
    {'id':'Q18','text':'What is the relationship between VaR and stress testing methodologies?','relevant':['DOC-05','DOC-14']},
    {'id':'Q19','text':'How does corporate valuation relate to capital structure decisions?','relevant':['DOC-06']},
    {'id':'Q20','text':'What role do time series models play in risk forecasting?','relevant':['DOC-05','DOC-07']},
    
    # ── Three Document Questions ──────────────────────────────────────────
    {'id':'Q21','text':'How do credit risk, market risk, and operational risk interact in banking?','relevant':['DOC-01','DOC-05','DOC-14']},
    {'id':'Q22','text':'What is the relationship between portfolio theory, derivatives, and risk management?','relevant':['DOC-02','DOC-04','DOC-05']},
    {'id':'Q23','text':'How do trading strategies, market microstructure, and machine learning intersect?','relevant':['DOC-03','DOC-05','DOC-08']},
    {'id':'Q24','text':'What connections exist between econometric models, risk measures, and forecasting?','relevant':['DOC-05','DOC-07','DOC-08']},
    {'id':'Q25','text':'How do behavioral biases affect portfolio decisions and market efficiency?','relevant':['DOC-02','DOC-03','DOC-09']},
    
    # ── Corporate Finance & Valuation ─────────────────────────────────────
    {'id':'Q26','text':'How is the Weighted Average Cost of Capital (WACC) calculated?','relevant':['DOC-06']},
    {'id':'Q27','text':'What factors influence corporate capital structure decisions?','relevant':['DOC-06']},
    {'id':'Q28','text':'How does M&A valuation incorporate risk assessment?','relevant':['DOC-05','DOC-06']},
    {'id':'Q29','text':'What is the relationship between corporate finance and financial regulation?','relevant':['DOC-06','DOC-13']},
    
    # ── Econometrics & Forecasting ────────────────────────────────────────
    {'id':'Q30','text':'What is a GARCH model and when is it used in finance?','relevant':['DOC-07']},
    {'id':'Q31','text':'Explain cointegration and its applications in pairs trading.','relevant':['DOC-07']},
    {'id':'Q32','text':'How do ARIMA models forecast financial time series?','relevant':['DOC-07']},
    {'id':'Q33','text':'What is the connection between econometric models and machine learning?','relevant':['DOC-07','DOC-08']},
    
    # ── Machine Learning in Finance ───────────────────────────────────────
    {'id':'Q34','text':'What are the applications of machine learning in financial prediction?','relevant':['DOC-08']},
    {'id':'Q35','text':'How do neural networks improve trading strategy performance?','relevant':['DOC-08']},
    {'id':'Q36','text':'How can ML enhance credit risk assessment in banking?','relevant':['DOC-01','DOC-08']},
    {'id':'Q37','text':'What role does AI play in portfolio optimization and risk management?','relevant':['DOC-02','DOC-05','DOC-08']},
    
    # ── Behavioral Finance ────────────────────────────────────────────────
    {'id':'Q38','text':'What are common behavioral biases that affect investor decisions?','relevant':['DOC-09']},
    {'id':'Q39','text':'Explain the concept of herding behavior in financial markets.','relevant':['DOC-09']},
    {'id':'Q40','text':'How do behavioral factors influence market microstructure?','relevant':['DOC-03','DOC-09']},
    
    # ── Cryptocurrency & Blockchain ───────────────────────────────────────
    {'id':'Q41','text':'How does blockchain technology enable cryptocurrency transactions?','relevant':['DOC-10']},
    {'id':'Q42','text':'What are the main differences between Bitcoin and Ethereum?','relevant':['DOC-10']},
    {'id':'Q43','text':'How do traditional risk management principles apply to crypto markets?','relevant':['DOC-05','DOC-10']},
    
    # ── ESG & Sustainable Finance ─────────────────────────────────────────
    {'id':'Q44','text':'How do ESG factors impact investment performance?','relevant':['DOC-11']},
    {'id':'Q45','text':'What is green finance and how does it promote sustainability?','relevant':['DOC-11']},
    {'id':'Q46','text':'How does ESG integration affect portfolio construction?','relevant':['DOC-02','DOC-11']},
    
    # ── Monetary Policy & Regulation ──────────────────────────────────────
    {'id':'Q47','text':'How do central banks use interest rates to control inflation?','relevant':['DOC-12']},
    {'id':'Q48','text':'What is quantitative easing and when is it implemented?','relevant':['DOC-12']},
    {'id':'Q49','text':'How do monetary policy decisions affect banking operations?','relevant':['DOC-01','DOC-12']},
    
    # ── Systemic Risk & Financial Stability ───────────────────────────────
    {'id':'Q50','text':'What is systemic risk and how is it measured?','relevant':['DOC-14']},
    {'id':'Q51','text':'Explain financial contagion and its transmission mechanisms.','relevant':['DOC-14']},
    {'id':'Q52','text':'How do regulatory frameworks address systemic risk?','relevant':['DOC-13','DOC-14']},
]

# Filter questions to only include those with documents in corpus
corpus_ids = {d['id'] for d in CORPUS}
QUESTIONS = [q for q in QUESTIONS if any(r in corpus_ids for r in q['relevant'])]

# Ensure we have exactly 50 questions (or as many as corpus supports)
QUESTIONS = QUESTIONS[:50]

print(f'✅ {len(QUESTIONS)} evaluation questions loaded.')
print(f'   Questions cover {len(set(r for q in QUESTIONS for r in q["relevant"]))} document topics')
print()

# Count questions by number of relevant documents
single_doc = sum(1 for q in QUESTIONS if len(q['relevant']) == 1)
two_docs = sum(1 for q in QUESTIONS if len(q['relevant']) == 2)
three_docs = sum(1 for q in QUESTIONS if len(q['relevant']) >= 3)

print('── Question distribution ──')
print(f'  Single document  : {single_doc} questions')
print(f'  Two documents    : {two_docs} questions')
print(f'  Three+ documents : {three_docs} questions')
print()

# Validate all relevant doc IDs exist in corpus
missing_any = False
for q in QUESTIONS:
    missing = [r for r in q['relevant'] if r not in corpus_ids]
    if missing:
        print(f'  ⚠️  {q["id"]} references unknown docs: {missing}')
        missing_any = True

if not missing_any:
    print('   All ground-truth document IDs validated ✓')


In [ ]:
# CELL 8 — Helper Functions
#
# Metrics:
#   Recall@K  — 1 if any relevant DOC appears in top-K retrieved chunks
#   MRR       — Mean Reciprocal Rank (1/rank of first relevant chunk by source doc)
#   P@5       — Precision@5 (kept for backward compatibility)
#
# Chunk-aware retrieval:
#   Chunks are indexed, but a chunk is counted as a HIT if its source_doc_id
#   matches a relevant document. Multiple chunks from the same doc count only once.

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer


def recall_at_k(retrieved_source_ids, relevant_ids, k):
    seen_docs = []
    for sid in retrieved_source_ids:
        if sid not in seen_docs:
            seen_docs.append(sid)
        if len(seen_docs) == k:
            break
    return 1.0 if any(d in relevant_ids for d in seen_docs) else 0.0


def reciprocal_rank(retrieved_source_ids, relevant_ids):
    seen_docs, rank = [], 0
    for sid in retrieved_source_ids:
        if sid not in seen_docs:
            seen_docs.append(sid)
            rank += 1
            if sid in relevant_ids:
                return 1.0 / rank
    return 0.0


def precision_at_k(retrieved_source_ids, relevant_ids, k):
    deduped = list(dict.fromkeys(retrieved_source_ids))[:k]
    return sum(1 for d in deduped if d in relevant_ids) / k


def load_model_with_fallback(model_key):
    cfg = MODELS[model_key]
    for hf_name in cfg['hf_chain']:
        try:
            print(f'    Trying: {hf_name} ...', flush=True)
            m = SentenceTransformer(hf_name, device=device)
            if hf_name != cfg['hf_chain'][0]:
                cfg['label'] += f' [fallback: {hf_name.split("/")[-1]}]'
                cfg['fallback_used'] = hf_name
            else:
                cfg['fallback_used'] = None
            cfg['loaded_name'] = hf_name
            print(f'    ✅ Loaded: {hf_name}')
            return m
        except Exception as e:
            print(f'    ⚠️  Failed ({type(e).__name__}: {str(e)[:60]})')
    raise RuntimeError(f'All fallbacks exhausted for {model_key}')


def build_chunk_collection(client, model_key, model):
    cfg       = MODELS[model_key]
    coll_name = cfg['collection']
    try:
        client.delete_collection(coll_name)
    except Exception:
        pass

    collection = client.create_collection(
        name=coll_name,
        metadata={'hnsw:space': 'cosine'},
    )

    texts     = [c['text']      for c in CHUNKS]
    ids       = [c['chunk_id']  for c in CHUNKS]
    metadatas = [{
        'source_doc_id': c['source_doc_id'],
        'source_title' : c['source_title'],
        'chunk_id'     : c['chunk_id'],
        'chunk_idx'    : c['chunk_idx'],
    } for c in CHUNKS]

    print(f'    Encoding {len(texts)} chunks ...', flush=True)
    t0         = time.time()
    embeddings = model.encode(texts, show_progress_bar=False, normalize_embeddings=True,
                              batch_size=64)
    print(f'    Encoded in {time.time()-t0:.1f}s | dim={embeddings.shape[1]}')

    # Add to ChromaDB in batches (max batch size = 5000)
    BATCH_SIZE = 5000
    print(f'    Adding {len(texts)} chunks to ChromaDB in batches of {BATCH_SIZE}...', flush=True)
    
    for i in range(0, len(texts), BATCH_SIZE):
        end_idx = min(i + BATCH_SIZE, len(texts))
        batch_texts = texts[i:end_idx]
        batch_embeddings = embeddings[i:end_idx].tolist()
        batch_ids = ids[i:end_idx]
        batch_metadatas = metadatas[i:end_idx]
        
        collection.add(
            documents=batch_texts,
            embeddings=batch_embeddings,
            ids=batch_ids,
            metadatas=batch_metadatas,
        )
        print(f'      Batch {i//BATCH_SIZE + 1}: Added {end_idx - i} chunks', flush=True)
    
    print(f'    ✓ All chunks added to collection')
    return collection


def run_queries_chunked(collection, model):
    results = []
    n_retrieve = TOP_K * 3  # over-fetch because dedup reduces count

    for q in QUESTIONS:
        q_emb = model.encode([q['text']], normalize_embeddings=True)[0].tolist()
        res   = collection.query(
            query_embeddings=[q_emb],
            n_results=min(n_retrieve, len(CHUNKS)),
            include=['documents', 'metadatas', 'distances'],
        )

        retrieved_chunks     = res['metadatas'][0]
        retrieved_source_ids = [m['source_doc_id'] for m in retrieved_chunks]
        retrieved_titles     = [m['source_title']  for m in retrieved_chunks]
        retrieved_chunk_ids  = [m['chunk_id']      for m in retrieved_chunks]
        distances            = [round(float(d), 4) for d in res['distances'][0]]

        results.append({
            'question_id'        : q['id'],
            'question_text'      : q['text'],
            'relevant_docs'      : q['relevant'],
            'retrieved_source_ids': retrieved_source_ids,
            'retrieved_chunk_ids' : retrieved_chunk_ids,
            'retrieved_titles'    : retrieved_titles,
            'distances'           : distances,
            'recall_at_1'         : recall_at_k(retrieved_source_ids, q['relevant'], 1),
            'recall_at_3'         : recall_at_k(retrieved_source_ids, q['relevant'], 3),
            'recall_at_5'         : recall_at_k(retrieved_source_ids, q['relevant'], 5),
            'precision_at_5'      : precision_at_k(retrieved_source_ids, q['relevant'], 5),
            'reciprocal_rank'     : reciprocal_rank(retrieved_source_ids, q['relevant']),
        })

    metrics = {
        'recall_at_1'   : float(np.mean([r['recall_at_1']    for r in results])),
        'recall_at_3'   : float(np.mean([r['recall_at_3']    for r in results])),
        'recall_at_5'   : float(np.mean([r['recall_at_5']    for r in results])),
        'precision_at_5': float(np.mean([r['precision_at_5'] for r in results])),
        'mrr'           : float(np.mean([r['reciprocal_rank'] for r in results])),
    }
    return results, metrics


def save_model_results(model_key, results, metrics):
    cfg  = MODELS[model_key]
    stem = model_key.lower()

    # JSON
    payload = {
        'model_key'    : model_key,
        'model_label'  : cfg['label'],
        'loaded_name'  : cfg.get('loaded_name', ''),
        'fallback_used': cfg.get('fallback_used'),
        'chunk_strategy': CHUNK_STRATEGY,
        'n_chunks'     : len(CHUNKS),
        'metrics'      : {k: round(v, 4) for k, v in metrics.items()},
        'timestamp'    : datetime.now().isoformat(),
        'results'      : results,
    }
    with open(OUTPUT_DIR / f'{stem}_results.json', 'w') as f:
        json.dump(payload, f, indent=2)

    # Human-readable TXT
    lines = []
    lines += ['=' * 80,
              f'  EXP-09 v3 RESULTS — {cfg["label"]}',
              f'  Model loaded : {cfg.get("loaded_name","?")}',
              f'  Fallback used: {cfg.get("fallback_used") or "None"}',
              f'  Chunking     : {CHUNK_STRATEGY.upper()} | {len(CHUNKS)} chunks from {len(CORPUS)} docs',
              '',
              f'  Recall@1 : {metrics["recall_at_1"]:.4f}  ({metrics["recall_at_1"]*100:.1f}%)',
              f'  Recall@3 : {metrics["recall_at_3"]:.4f}  ({metrics["recall_at_3"]*100:.1f}%)',
              f'  Recall@5 : {metrics["recall_at_5"]:.4f}  ({metrics["recall_at_5"]*100:.1f}%)',
              f'  MRR      : {metrics["mrr"]:.4f}',
              f'  P@5      : {metrics["precision_at_5"]:.4f}',
              f'  Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
              '=' * 80]

    for r in results:
        rr = r['reciprocal_rank']
        if rr == 1.0:   hit_mark = '★ RANK-1 HIT'
        elif rr > 0:    hit_mark = f'  rank-{int(round(1/rr))} hit'
        else:           hit_mark = '  MISS'
        lines += [f'\n{" - "*26}',
                  f'  [{r["question_id"]}] R@1={r["recall_at_1"]:.0f}  '
                  f'R@3={r["recall_at_3"]:.0f}  R@5={r["recall_at_5"]:.0f}  '
                  f'RR={rr:.3f}  {hit_mark}',
                  f'  Q: {textwrap.fill(r["question_text"], 66, subsequent_indent="     ")}',
                  f'  Relevant docs: {", ".join(r["relevant_docs"])}']

        seen_docs = []
        for chunk_id, src_id, title, dist in zip(
                r['retrieved_chunk_ids'][:TOP_K],
                r['retrieved_source_ids'][:TOP_K],
                r['retrieved_titles'][:TOP_K],
                r['distances'][:TOP_K]):
            rank_in_dedup = len([d for d in seen_docs if True]) + 1
            marker = '◉' if src_id in r['relevant_docs'] else '○'
            if src_id not in seen_docs:
                seen_docs.append(src_id)
            lines.append(f'    {rank_in_dedup}. {marker} [{chunk_id}→{src_id}] '
                         f'{title[:45]}  (dist={dist})')

    with open(OUTPUT_DIR / f'{stem}_results.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))

    return OUTPUT_DIR / f'{stem}_results.txt'


print('✅ Helper functions ready.')
print(f'   Metrics: Recall@1, Recall@3, Recall@5, MRR, P@5')
print(f'   Chunk-aware: chunk hits mapped back to source documents')


In [ ]:
# CELL 9 — Run the Experiment
print('=' * 70)
print('  EXP-09 v3  DOMAIN-SPECIFIC EMBEDDINGS BENCHMARK')
print(f'  Dataset   : arXiv Quantitative Finance (q-fin) Research Papers')
print(f'  Corpus    : {len(CORPUS)} topic documents → {len(CHUNKS)} {CHUNK_STRATEGY} chunks')
print(f'  Questions : {len(QUESTIONS)}')
print(f'  Metrics   : Recall@1, Recall@3, Recall@5, MRR, P@5')
print('=' * 70)

chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH,
    settings=Settings(anonymized_telemetry=False),
)

all_results = {}

for model_key, cfg in MODELS.items():
    print(f'\n{"-"*70}')
    print(f'  [{model_key}]  {cfg["label"]}')
    print(f'  Fallback chain: {" → ".join(cfg["hf_chain"])}')
    print(f'{"-"*70}')

    t0 = time.time()
    model = load_model_with_fallback(model_key)

    collection = build_chunk_collection(chroma_client, model_key, model)

    print(f'    Running {len(QUESTIONS)} queries ...', flush=True)
    t1 = time.time()
    results, metrics = run_queries_chunked(collection, model)
    print(f'    Queries done in {time.time()-t1:.1f}s')
    print(f'    ★  R@1={metrics["recall_at_1"]*100:.1f}%  '
          f'R@3={metrics["recall_at_3"]*100:.1f}%  '
          f'R@5={metrics["recall_at_5"]*100:.1f}%  '
          f'MRR={metrics["mrr"]:.3f}  '
          f'P@5={metrics["precision_at_5"]*100:.1f}%')

    txt_path = save_model_results(model_key, results, metrics)
    print(f'    Saved → {txt_path}')
    all_results[model_key] = (results, metrics)
    print(f'    Total time: {time.time()-t0:.1f}s')

# ── Quick Final Leaderboard ───────────────────────────────────────────────────
print('\n' + '='*70)
print('  FINAL RANKING (by MRR)')
print('='*70)
ranked = sorted(all_results.items(), key=lambda x: x[1][1]['mrr'], reverse=True)
for rank, (mk, (_, m)) in enumerate(ranked, 1):
    bar = '█' * int(m['mrr'] * 30)
    fb  = ' [FALLBACK]' if MODELS[mk].get('fallback_used') else ''
    print(f'  #{rank}  {bar:<30}  MRR={m["mrr"]:.3f}  '
          f'R@1={m["recall_at_1"]*100:.0f}%  R@3={m["recall_at_3"]*100:.0f}%  '
          f'R@5={m["recall_at_5"]*100:.0f}%  {MODELS[mk]["label"]}{fb}')
print()
print(f'✅ All models evaluated.')


In [ ]:
# CELL 10 — Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec

plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.spines.top': False,
                     'axes.spines.right': False})

# Colour scheme: blues = general, reds/oranges = domain-specific
PALETTE = {
    'A_MiniLM' : '#2563eb',
    'B_BGE'    : '#60a5fa',
    'C_FinBERT': '#f97316',
    'D_FinNews': '#dc2626',
}
mk_order   = list(MODELS.keys())
short_lbl  = [MODELS[mk]['label'].split('(')[0].strip()[:22] for mk in mk_order]
colors     = [PALETTE[mk] for mk in mk_order]

# ── Figure 1: 4-panel metric bar chart ───────────────────────────────────────
metric_defs = [
    ('recall_at_1',    'Recall@1',    'Did the top result come\nfrom the right document?'),
    ('recall_at_3',    'Recall@3',    'Right doc in top-3\nchunks retrieved?'),
    ('recall_at_5',    'Recall@5',    'Right doc in top-5\nchunks retrieved?'),
    ('mrr',            'MRR × 100',  'Mean Reciprocal Rank\n(higher = better ordering)'),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
fig.suptitle(
    f'EXP-09 v3 — Domain-Specific Embeddings Benchmark\n'
    f'Dataset: arXiv Finance Papers  |  Chunking: {CHUNK_STRATEGY.upper()}  |  '
    f'{len(CHUNKS)} chunks from {len(CORPUS)} docs',
    fontsize=12, fontweight='bold', y=1.02
)

for ax, (metric_key, label, subtitle) in zip(axes, metric_defs):
    scores = [all_results[mk][1][metric_key] * 100 for mk in mk_order]
    bars = ax.bar(short_lbl, scores, color=colors, edgecolor='white', linewidth=0.8,
                  width=0.55)
    ax.set_title(label, fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel(subtitle, fontsize=8, color='#555')
    ax.set_ylim(0, 108)
    ax.tick_params(axis='x', rotation=20, labelsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
    ax.grid(axis='y', alpha=0.25, linewidth=0.5)
    ax.set_axisbelow(True)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f'{score:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

blue_p   = mpatches.Patch(color='#2563eb', label='General-purpose models')
orange_p = mpatches.Patch(color='#f97316', label='Finance domain-specific models')
fig.legend(handles=[blue_p, orange_p], loc='lower center', ncol=2, fontsize=10,
           frameon=False, bbox_to_anchor=(0.5, -0.06))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp09v3_bar_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Bar chart saved.')

# ── Figure 2: Per-question MRR line chart ────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(17, 4.5))
q_ids = [q['id'] for q in QUESTIONS]
x     = range(len(QUESTIONS))

for mk in mk_order:
    rr_vals = [all_results[mk][0][i]['reciprocal_rank'] for i in range(len(QUESTIONS))]
    label   = MODELS[mk]['label'].split('(')[0].strip()[:22]
    fb      = ' ↺' if MODELS[mk].get('fallback_used') else ''
    ax2.plot(x, rr_vals, label=label + fb, color=PALETTE[mk],
             linewidth=1.8, alpha=0.85, marker='o', markersize=3)

ax2.set_title('Per-Question Reciprocal Rank  '
              '(1.0 = rank-1 hit · 0.5 = rank-2 · 0.0 = miss)', fontsize=11)
ax2.set_xlabel('Question ID')
ax2.set_ylabel('Reciprocal Rank')
ax2.set_xticks(range(len(QUESTIONS)))
ax2.set_xticklabels(q_ids, rotation=45, fontsize=7)
ax2.set_ylim(-0.05, 1.1)
ax2.axhline(0.5, color='#ccc', linewidth=0.8, linestyle='--')
ax2.legend(fontsize=9, loc='lower right')
ax2.grid(axis='y', alpha=0.2)
ax2.set_axisbelow(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp09v3_per_question_rr.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Per-question RR chart saved.')

# ── Figure 3: Domain vs General delta heatmap ────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(17, 3.5))
general_keys = ['A_MiniLM', 'B_BGE']
domain_keys  = ['C_FinBERT', 'D_FinNews']

deltas = []
for i in range(len(QUESTIONS)):
    gen_rr = np.mean([all_results[mk][0][i]['reciprocal_rank'] for mk in general_keys])
    dom_rr = np.mean([all_results[mk][0][i]['reciprocal_rank'] for mk in domain_keys])
    deltas.append(dom_rr - gen_rr)

colors_heat = ['#dc2626' if d >= 0 else '#2563eb' for d in deltas]
ax3.bar(range(len(QUESTIONS)), deltas, color=colors_heat, alpha=0.8, width=0.7)
ax3.axhline(0, color='#333', linewidth=0.8)
ax3.set_title('Domain vs General: Δ MRR per question  '
              '(red = domain wins · blue = general wins)', fontsize=11)
ax3.set_xticks(range(len(QUESTIONS)))
ax3.set_xticklabels(q_ids, rotation=45, fontsize=7)
ax3.set_ylabel('Δ Reciprocal Rank')
ax3.grid(axis='y', alpha=0.2)
ax3.set_axisbelow(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp09v3_domain_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Domain delta chart saved.')


In [ ]:
# CELL 11 — Professional Summary Report (Markdown + JSON)

def build_summary_report(all_results):
    ranked = sorted(all_results.items(), key=lambda x: (x[1][1]['mrr'], x[1][1]['recall_at_3']), reverse=True)
    winner_key, (_, winner_m) = ranked[0]

    general_keys = ['A_MiniLM', 'B_BGE']
    domain_keys  = ['C_FinBERT', 'D_FinNews']

    gen_mrr_avg = np.mean([all_results[mk][1]['mrr'] for mk in general_keys])
    dom_mrr_avg = np.mean([all_results[mk][1]['mrr'] for mk in domain_keys])
    domain_wins = dom_mrr_avg >= gen_mrr_avg
    delta_pct   = abs(dom_mrr_avg - gen_mrr_avg) / max(gen_mrr_avg, 1e-9) * 100

    fallback_notes = []
    for mk, cfg in MODELS.items():
        if cfg.get('fallback_used'):
            fallback_notes.append(
                f'- **{mk}** (`{cfg["hf_chain"][0]}`): primary unavailable → '
                f'fell back to `{cfg["fallback_used"]}`'
            )

    # Per-question domain delta
    q_deltas = []
    for i, q in enumerate(QUESTIONS):
        gen_rr = np.mean([all_results[mk][0][i]['reciprocal_rank'] for mk in general_keys])
        dom_rr = np.mean([all_results[mk][0][i]['reciprocal_rank'] for mk in domain_keys])
        q_deltas.append((q['id'], q['text'], gen_rr, dom_rr, dom_rr - gen_rr))

    top5_domain = sorted(q_deltas, key=lambda x: x[4], reverse=True)[:5]
    top5_general = sorted(q_deltas, key=lambda x: x[4])[:5]

    lines = []
    lines += [
        '# EXP-09 v3 — Embedding Benchmark Summary Report',
        '',
        f'**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M")}',
        f'**Dataset:** arXiv Quantitative Finance (q-fin) — '
        f'Full-text research papers from [arXiv.org](https://arxiv.org/list/q-fin/recent)',
        f'**Corpus:** {len(CORPUS)} topic documents → {len(CHUNKS)} chunks '
        f'({CHUNK_STRATEGY.upper()} chunking · target {CHUNK_SIZE} chars · {CHUNK_OVERLAP} char overlap)',
        f'**Questions:** {len(QUESTIONS)} evaluation queries with ground-truth source documents',
        '',
        '---',
        '',
        '## 1. Model Rankings',
        '',
        f'| Rank | Model | Recall@1 | Recall@3 | Recall@5 | MRR | Fallback? |',
        f'|------|-------|----------|----------|----------|-----|-----------|',
    ]

    for rank, (mk, (_, m)) in enumerate(ranked, 1):
        cfg = MODELS[mk]
        fb  = f'→ `{cfg["fallback_used"].split("/")[-1]}`' if cfg.get('fallback_used') else '—'
        medal = ['🥇', '🥈', '🥉', '  '][min(rank-1, 3)]
        lines.append(
            f'| {medal} #{rank} | **{cfg["label"]}** | '
            f'{m["recall_at_1"]*100:.1f}% | {m["recall_at_3"]*100:.1f}% | '
            f'{m["recall_at_5"]*100:.1f}% | **{m["mrr"]:.3f}** | {fb} |'
        )

    lines += [
        '',
        '> **Primary metric: MRR** (Mean Reciprocal Rank). Rewards rank-1 hits more than rank-5.',
        '> **Recall@3** is the practical RAG window — most pipelines send top-3 chunks to the LLM.',
        '',
        '---',
        '',
        '## 2. Dataset & Corpus',
        '',
        '**Source:** arXiv Quantitative Finance (q-fin) Research Papers — Full-text academic papers',
        'covering banking, portfolio management, risk management, derivatives, and other finance topics.',
        '',
        '**Corpus construction:**',
        '1. Downloaded ~40 full-text research papers from arXiv q-fin category.',
        '2. Papers were classified into finance topic categories using keyword matching.',
        '3. Papers on the same topic were combined into thematic documents.',
        '4. Each document was chunked using the **' + CHUNK_STRATEGY.upper() + '** strategy.',
        '',
        '| Topic | Papers | Chunks |',
        '|-------|--------|--------|',
    ]

    chunk_map = {}
    for c in CHUNKS:
        chunk_map.setdefault(c['source_doc_id'], 0)
        chunk_map[c['source_doc_id']] += 1

    for d in CORPUS:
        lines.append(f'| {d["id"]} {d["title"]} | {d["paper_count"]} | {chunk_map.get(d["id"],0)} |')

    lines += [
        '',
        '---',
        '',
        '## 3. Chunking Strategy',
        '',
        f'**Strategy used: {CHUNK_STRATEGY.upper()}**',
        '',
        '| Strategy | Description | When to prefer |',
        '|----------|-------------|----------------|',
        '| **Recursive** | Splits on `\\n\\n` → `\\n` → `.` → space → char; fixed window with overlap | Speed, reproducibility |',
        '| **Semantic** | Sentence boundary detection + cosine similarity gate; new chunk when similarity < threshold | Quality, coherence |',
        '',
        f'- Total chunks: **{len(CHUNKS)}** from {len(CORPUS)} documents',
        f'- Avg chunk size: **{np.mean([len(c["text"]) for c in CHUNKS]):.0f} chars**',
        f'- Semantic threshold: {SEM_THRESHOLD} (cosine sim below this starts a new chunk)',
        '',
        '**Why use full research papers:** Full academic papers provide comprehensive, detailed',
        'content covering methodology, results, and analysis - ideal for testing RAG retrieval',
        'on complex, technical finance topics.',
        '',
        '**Why v2 had no chunking:** v2 embedded whole documents as single vectors.',
        'With long finance docs, a single vector averages over many sub-topics and loses',
        'specificity. Chunking creates granular, semantically focused embeddings that',
        'match narrower question intents far more precisely.',
        '',
        '---',
        '',
        '## 4. Domain vs General Embedding Models',
        '',
    ]

    if domain_wins:
        lines += [
            f'**Domain-specific models outperform general models** (avg MRR: domain={dom_mrr_avg:.3f} vs general={gen_mrr_avg:.3f}, Δ={delta_pct:.1f}%)',
            '',
            'Finance-trained embeddings better capture domain terminology (NIM, LCR, Basel III, CVA, SOFR),',
            'allowing them to associate technical questions with the correct document chunks even when',
            'surface-level lexical overlap is low.',
        ]
    else:
        lines += [
            f'**General models are competitive** (avg MRR: general={gen_mrr_avg:.3f} vs domain={dom_mrr_avg:.3f})',
            '',
            'This can occur when: (1) domain model was loaded via fallback and the fallback model',
            'was not fine-tuned on finance-similarity tasks; (2) the corpus sentences are drawn from',
            'news wire text, which general models have seen extensively; (3) the query set tests',
            'surface-level matching rather than deep domain semantics.',
        ]

    lines += [
        '',
        '**Queries where domain models win most:**',
        '',
    ]
    for qid, qtxt, gen_rr, dom_rr, delta in top5_domain:
        lines.append(f'- **{qid}** Δ={delta:+.3f} (domain={dom_rr:.3f} vs general={gen_rr:.3f})')
        lines.append(f'  *{qtxt[:90]}*')

    lines += [
        '',
        '**Queries where general models win most:**',
        '',
    ]
    for qid, qtxt, gen_rr, dom_rr, delta in top5_general:
        lines.append(f'- **{qid}** Δ={delta:+.3f} (general={gen_rr:.3f} vs domain={dom_rr:.3f})')
        lines.append(f'  *{qtxt[:90]}*')

    lines += ['', '---', '', '## 5. Fallback Model Notes', '']
    if fallback_notes:
        lines += fallback_notes
    else:
        lines.append('No fallbacks were triggered — all primary models loaded successfully.')

    lines += [
        '',
        '---',
        '',
        '## 6. Recommendations',
        '',
        f'1. **Best model for production:** {MODELS[winner_key]["label"]} '
        f'(MRR = {winner_m["mrr"]:.3f})',
        f'2. **Chunking:** Apply {CHUNK_STRATEGY.upper()} chunking before indexing — '
        'it yields more precise chunk-level retrieval than whole-document embedding.',
        '3. **Corpus quality matters:** Replacing hand-crafted demo documents with real financial',
        '   news sentences improves retrieval realism and benchmark validity.',
        '4. **FinBERT note:** `yiyanghkust/finbert-tone` is a _sentiment classifier_, not a',
        '   sentence similarity model. Its embeddings may underperform on retrieval tasks.',
        '   Consider `nickmuchi/financial-news-distilroberta-base` as primary finance model.',
        '5. **Scale up:** For production RAG, combine the best embedding model with a',
        '   bi-encoder → cross-encoder reranking pipeline for maximum precision.',
        '',
        '---',
        '',
        '## 7. Output Files',
        '',
        '```',
        'exp09_outputs_v3/',
        '├── a_minilm_results.txt / .json',
        '├── b_bge_results.txt / .json',
        '├── c_finbert_results.txt / .json',
        '├── d_finnews_results.txt / .json',
        '├── exp09v3_bar_metrics.png       ← 4-panel bar chart',
        '├── exp09v3_per_question_rr.png   ← per-question line chart',
        '├── exp09v3_domain_delta.png      ← domain vs general delta',
        '└── exp09v3_summary_report.md     ← this report',
        '```',
    ]

    report_text = '\n'.join(lines)
    report_path = OUTPUT_DIR / 'exp09v3_summary_report.md'
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(report_text)

    # Also save comparison JSON
    comparison = {
        'experiment'    : 'EXP-09-v3',
        'dataset'       : 'arXiv Quantitative Finance (q-fin) Research Papers',
        'chunk_strategy': CHUNK_STRATEGY,
        'n_chunks'      : len(CHUNKS),
        'n_documents'   : len(CORPUS),
        'n_questions'   : len(QUESTIONS),
        'timestamp'     : datetime.now().isoformat(),
        'ranking'       : [
            {
                'rank'         : i+1,
                'model_key'    : mk,
                'model_label'  : MODELS[mk]['label'],
                'loaded_name'  : MODELS[mk].get('loaded_name', ''),
                'fallback_used': MODELS[mk].get('fallback_used'),
                'metrics'      : {k: round(v, 4) for k, v in m.items()},
            }
            for i, (mk, (_, m)) in enumerate(ranked)
        ],
        'per_question': {
            q['id']: {
                mk: {
                    'r1': all_results[mk][0][qi]['recall_at_1'],
                    'r3': all_results[mk][0][qi]['recall_at_3'],
                    'r5': all_results[mk][0][qi]['recall_at_5'],
                    'rr': round(all_results[mk][0][qi]['reciprocal_rank'], 3),
                }
                for mk in MODELS
            }
            for qi, q in enumerate(QUESTIONS)
        },
    }
    with open(OUTPUT_DIR / 'exp09v3_comparison.json', 'w') as f:
        json.dump(comparison, f, indent=2)

    return report_path, report_text

report_path, report_text = build_summary_report(all_results)
print(report_text)
print(f'\n✅ Summary report saved → {report_path}')


In [ ]:
# CELL 12 — Download All Outputs as ZIP
import shutil
from google.colab import files

zip_path = '/content/exp09_v3_outputs'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f'✅ ZIP created: {zip_path}.zip')
print(f'   Contents of {OUTPUT_DIR}:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'     {f.name:50s}  {f.stat().st_size / 1024:6.1f} KB')

files.download(f'{zip_path}.zip')
print('\n📥 Download started. Also accessible in Files panel → /content/exp09_outputs_v3/')
